# Banking Intent Classifier: Portfolio Project
## Exploring the ([CRISP-DM Framework](https://www.datascience-pm.com/crisp-dm-2/))

## Phase 1: Business Understanding

**Goal:** Automate classification of customer banking queries into 77 predefined text categories to improve routing speed, reduce manual triage, and increase first response accuracy. 

**Models being used:**
- Baseline: Multi-Layer Perceptron (MLP) using TF-IDF features
- Advanced: RoBERTa finetuned with LoRA (Low-Rank Adaptation)

## Phase 2 & 3: Data Understanding & Exploratory Data Analysis

**Data:** Banking77 dataset from codeacademy.

I will do some exploratory data analysis on the Banking77 dataset to understand its contents before doing any ML.

**Steps:**
- Load and verify both the training and test datasets
- Inspect class distribution across all 77 text categories
- Analyze query text lengths across text classes
- Identify common and distringuishing words per text class
- Document key findings to inform the preprocessing and modeling strategy


In [1]:
# Clone repo into Colab's server
import os, subprocess
repo_path = "/content/banking-intent-classifier"
if not os.path.isdir(repo_path):
    subprocess.run(["git", "clone", "https://github.com/ovesa/banking-intent-classifier", repo_path], check=True)
else:
    subprocess.run(["git", "-C", repo_path, "pull"], check=True)

os.chdir(repo_path)

# Install dependencies
%pip install -r requirements.txt
%pip install peft

  Using cached numpy-2.2.6-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
  Using cached pandas-2.3.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (91 kB)
  Using cached scipy-1.15.3-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached scikit_learn-1.7.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (11 kB)
  Using cached sympy-1.13.3-py3-none-any.whl.metadata (12 kB)
  Using cached numexpr-2.10.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (1.2 kB)
  Using cached joblib-1.4.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached cloudpickle-3.1.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached matplotlib-3.10.7-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (11 kB)
  Using cached networkx-3.4.2-py3-none-any.whl.metadata (6.3 kB)
  Using cached torch-2.4.1-cp312-cp312-manylinux1_x86_64.whl.metadata (26 kB)
  Using cached torchvisio

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from collections import Counter
import re

from plot_style import set_plot_style
set_plot_style()
sns.set_theme(style="whitegrid", palette="muted")

In [3]:
# Load the data
train_df = pd.read_csv("datasets/banking77_train.csv")
test_df = pd.read_csv("datasets/banking77_test.csv")

# Verification of the data
print(f"Train set shape: {train_df.shape}")
print(f"Train set columns: {train_df.columns}")
print(f"Missing values in train set:\n{train_df.isnull().sum()}\n")
print(f"Unique text in train set: {train_df['text'].nunique()}")

print(f"Test set shape: {test_df.shape}")
print(f"Test set columns: {test_df.columns}")
print(f"Missing values in test set:\n{test_df.isnull().sum()}\n")
print(f"Unique text in test set: {test_df['text'].nunique()}")

print("Sample data")
train_df.head(6)

Train set shape: (10003, 2)
Train set columns: Index(['text', 'category'], dtype='object')
Missing values in train set:
text        0
category    0
dtype: int64

Unique text in train set: 10003
Test set shape: (3080, 2)
Test set columns: Index(['text', 'category'], dtype='object')
Missing values in test set:
text        0
category    0
dtype: int64

Unique text in test set: 3080
Sample data


,text,category
0,I am still waiting on my card?,card_arrival
1,What can I do if my card still hasn't arrived ...,card_arrival
2,I have been waiting over a week. Is the card s...,card_arrival
3,Can I track my card while it is in the process...,card_arrival
4,"How do I know if I will get my card, or if it ...",card_arrival
5,When did you send me my new card?,card_arrival


In [4]:
# verification of category distribution
print(f"Unique text classes: {train_df['category'].nunique()}")
print("Samples per class (train):")
print(train_df['category'].value_counts().describe())

print("Samples per class (test):")
print(test_df['category'].value_counts().describe())

Unique text classes: 77
Samples per class (train):
count     77.000000
mean     129.909091
std       32.942207
min       35.000000
25%      112.000000
50%      127.000000
75%      159.000000
max      187.000000
Name: count, dtype: float64
Samples per class (test):
count    77.0
mean     40.0
std       0.0
min      40.0
25%      40.0
50%      40.0
75%      40.0
max      40.0
Name: count, dtype: float64


### Finding: Class Distribution

- 77 classes are seen in both training and test datast
- Training set has about ~130 samples per class on average
- Test set is balanced at 40 samples per class
- No class imbalance

In [ ]:
# Visualize the distribution of text classes in the training set

text_counts = train_df['text'].value_counts().sort_values()

fig, ax = plt.subplots(figsize=(10, 18))
text_counts.plot(kind='barh', ax=ax, color='dodgerblue', edgecolor='k')

ax.set_xlabel("Number of Samples")
ax.set_title("Training Text Class Distribution")
plt.tight_layout()
plt.show()